# ScentAI Clean Retrieval - Stage 2

This notebook validates only the retrieval boundary:

`structured request -> BGE-M3 + Chroma + community graph -> grounded candidates`

It does not install, import, start, or call vLLM. Retrieval runs in its own
CPU-only uv environment and exposes a localhost HTTP API. Use a fresh Colab
runtime and **Run all**. A GPU is not required for this stage.

In [ ]:
import shutil
import subprocess
import sys


def run_checked(command, *, label):
    print(f"\n[{label}]", " ".join(map(str, command)))
    subprocess.run([str(part) for part in command], check=True)


run_checked(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "uv"],
    label="install uv launcher",
)
UV = shutil.which("uv")
assert UV, "uv was installed but its executable is not on PATH."
print(subprocess.check_output([UV, "--version"], text=True).strip())

## Create a CPU-only retrieval environment

Only two top-level packages are requested. uv resolves their compatible NumPy,
SciPy, sklearn, Transformers, and CPU PyTorch versions inside this environment.
Nothing from this environment is imported into the notebook kernel.

In [ ]:
from pathlib import Path

RETRIEVAL_ENV = Path("/content/scentai_retrieval_env")
RETRIEVAL_PYTHON = RETRIEVAL_ENV / "bin" / "python"

run_checked(
    [
        UV, "venv", "--clear", "--python", "3.12", "--seed", "--managed-python",
        str(RETRIEVAL_ENV),
    ],
    label="create clean retrieval environment",
)
run_checked(
    [
        UV, "pip", "install", "--python", str(RETRIEVAL_PYTHON),
        "chromadb==1.5.9", "sentence-transformers==5.6.0",
        "--torch-backend=cpu", "--strict",
    ],
    label="install retrieval dependencies with CPU PyTorch",
)
run_checked(
    [UV, "pip", "check", "--python", str(RETRIEVAL_PYTHON)],
    label="check retrieval dependency graph",
)
assert RETRIEVAL_PYTHON.exists(), RETRIEVAL_PYTHON

## Dependency smoke test

In [ ]:
import json

probe_code = r'''import json
from importlib.metadata import version
import chromadb
import numpy
import scipy
import sklearn
import sentence_transformers
import torch
import transformers

print("SCENTAI_RETRIEVAL_PROBE=" + json.dumps({
    "chromadb": version("chromadb"),
    "sentence_transformers": version("sentence-transformers"),
    "transformers": version("transformers"),
    "torch": version("torch"),
    "torch_cuda": torch.version.cuda,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
}))
'''
probe = subprocess.run(
    [str(RETRIEVAL_PYTHON), "-c", probe_code],
    text=True,
    capture_output=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("The isolated retrieval environment failed its import smoke test.")
probe_line = next(
    (line for line in probe.stdout.splitlines() if line.startswith("SCENTAI_RETRIEVAL_PROBE=")),
    None,
)
assert probe_line, probe.stdout
retrieval_versions = json.loads(probe_line.split("=", 1)[1])
print(json.dumps(retrieval_versions, indent=2))
assert retrieval_versions["chromadb"] == "1.5.9", retrieval_versions
assert retrieval_versions["sentence_transformers"] == "5.6.0", retrieval_versions
assert retrieval_versions["torch_cuda"] is None, (
    "Retrieval must use CPU-only PyTorch so it cannot reserve inference VRAM."
)
print("Retrieval dependency smoke test: passed")

## Mount Drive and stage immutable data locally

Reading the 1.5 GB HNSW index directly from Drive is unnecessarily slow. The
notebook copies Chroma and the 94 MB SQLite catalog to Colab's local disk once.

In [ ]:
import os
import time
from google.colab import drive, userdata

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/Perfume-Dataset")
DRIVE_CHROMA_DIR = PROJECT_DIR / "chroma_db_bge_m3"
DRIVE_CATALOG = PROJECT_DIR / "scentai_catalog.sqlite3"
LOCAL_CHROMA_DIR = Path("/content/scentai_data/chroma_db_bge_m3")
LOCAL_CATALOG = Path("/content/scentai_data/scentai_catalog.sqlite3")

assert (DRIVE_CHROMA_DIR / "chroma.sqlite3").exists(), f"Missing Chroma DB: {DRIVE_CHROMA_DIR}"
assert DRIVE_CATALOG.exists(), f"Missing catalog: {DRIVE_CATALOG}"

copy_started = time.perf_counter()
if LOCAL_CHROMA_DIR.parent.exists():
    shutil.rmtree(LOCAL_CHROMA_DIR.parent)
LOCAL_CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_CHROMA_DIR, LOCAL_CHROMA_DIR)
shutil.copy2(DRIVE_CATALOG, LOCAL_CATALOG)
copy_elapsed = time.perf_counter() - copy_started

chroma_bytes = sum(path.stat().st_size for path in LOCAL_CHROMA_DIR.rglob("*") if path.is_file())
print(f"Local Chroma: {chroma_bytes / (1024 ** 3):.2f} GB")
print(f"Local catalog: {LOCAL_CATALOG.stat().st_size / (1024 ** 2):.1f} MB")
print(f"Local staging time: {copy_elapsed:.1f}s")
assert chroma_bytes > 1_000_000_000, "The copied Chroma index looks incomplete."
assert LOCAL_CATALOG.stat().st_size > 50_000_000, "The copied catalog looks incomplete."

try:
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    HF_TOKEN = ""

## Materialize and start the isolated retrieval service

In [ ]:
SERVICE_SOURCE = 'from __future__ import annotations\n\nimport difflib\nimport json\nimport math\nimport os\nimport re\nimport sqlite3\nimport threading\nimport time\nimport unicodedata\nfrom collections import defaultdict\nfrom functools import lru_cache\nfrom http import HTTPStatus\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nfrom typing import Any\n\n\nBGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "\nDEFAULT_COLLECTION = "scentai_perfumes"\nDEFAULT_MODEL = "BAAI/bge-m3"\nVALID_GENDERS = {"male", "female", "unisex"}\nVALID_SEASONS = {"spring", "summer", "autumn", "winter"}\nVALID_TIMES = {"day", "night"}\nVALID_DISCOVERY_MODES = {"balanced", "mainstream", "niche"}\n\nCONCENTRATION_ALIAS_PATTERNS = (\n    (re.compile(r"\\be\\s*d\\s*p\\b|\\bepd\\b"), "eau de parfum"),\n    (re.compile(r"\\be\\s*d\\s*t\\b|\\betd\\b"), "eau de toilette"),\n    (re.compile(r"\\be\\s*d\\s*c\\b"), "eau de cologne"),\n    (re.compile(r"\\bparfume\\b"), "parfum"),\n)\nCONCENTRATION_SIGNATURES = (\n    ("extrait de parfum", "extrait"),\n    ("eau de parfum", "edp"),\n    ("eau de toilette", "edt"),\n    ("eau de cologne", "edc"),\n)\nCONCENTRATION_TOKENS = {"edp", "edt", "edc", "extrait", "parfum", "cologne"}\nEDITION_TOKENS = {\n    *CONCENTRATION_TOKENS,\n    "absolu", "absolute", "elixir", "extreme", "fraiche", "intense",\n    "limited", "edition", "oil", "sport", "spray",\n}\nIDENTITY_CONNECTORS = {"a", "by", "de", "des", "du", "for", "le", "la", "of", "pour", "the"}\n\n# These are spelling/market-name exceptions that cannot be inferred from initials\n# alone. Initialisms such as YSL, JPG, LV, MFK, PDM, and CDG are generated from\n# the catalog dynamically below.\nEXPLICIT_BRAND_ALIASES = {\n    "armani": "giorgio armani",\n    "cdg": "comme des garcons",\n    "ch": "carolina herrera",\n    "d g": "dolce gabbana",\n    "dg": "dolce gabbana",\n    "jpg": "jean paul gaultier",\n    "lv": "louis vuitton",\n    "mfk": "maison francis kurkdjian",\n    "pdm": "parfums de marly",\n    "tf": "tom ford",\n    "ysl": "yves saint laurent",\n}\n\n\ndef normalize_text(value: Any) -> str:\n    text = unicodedata.normalize("NFKD", str(value or "").lower())\n    text = "".join(char for char in text if not unicodedata.combining(char))\n    text = re.sub(r"[^a-z0-9]+", " ", text)\n    return " ".join(text.split())\n\n\ndef expand_concentration_aliases(value: Any) -> str:\n    """Expand common concentration abbreviations before identity matching."""\n    normalized = normalize_text(value)\n    for pattern, replacement in CONCENTRATION_ALIAS_PATTERNS:\n        normalized = pattern.sub(replacement, normalized)\n    return " ".join(normalized.split())\n\n\ndef identity_signature(value: Any) -> tuple[list[str], set[str]]:\n    """Return identity-bearing name tokens and explicit edition qualifiers."""\n    normalized = expand_concentration_aliases(value)\n    for phrase, marker in CONCENTRATION_SIGNATURES:\n        normalized = re.sub(rf"\\b{re.escape(phrase)}\\b", marker, normalized)\n    tokens = normalized.split()\n    qualifiers = {token for token in tokens if token in EDITION_TOKENS}\n    core = [\n        token for token in tokens\n        if token not in EDITION_TOKENS and token not in IDENTITY_CONNECTORS\n    ]\n    return core, qualifiers\n\n\ndef perfume_label(name: Any, brand: Any) -> str:\n    name_text = str(name or "").strip()\n    brand_text = str(brand or "").strip()\n    if not brand_text:\n        return name_text\n    normalized_name = normalize_text(name_text)\n    normalized_brand = normalize_text(brand_text)\n    if normalized_name == normalized_brand or normalized_name.endswith(f" by {normalized_brand}"):\n        return name_text\n    return f"{name_text} by {brand_text}"\n\n\ndef csv_terms(value: Any) -> set[str]:\n    return {normalize_text(part) for part in str(value or "").split(",") if normalize_text(part)}\n\n\ndef trait_variants(value: Any) -> set[str]:\n    """Return directional taxonomy variants for user-facing trait constraints."""\n    normalized = normalize_text(value)\n    return {\n        "musk": {"musk", "musky", "white musk"},\n        "musky": {"musk", "musky", "white musk"},\n        "leather": {"leather", "leathery", "suede"},\n        "leathery": {"leather", "leathery", "suede"},\n        "oud": {"oud", "aoud", "agarwood"},\n        "aoud": {"oud", "aoud", "agarwood"},\n        "agarwood": {"oud", "aoud", "agarwood"},\n        "smoke": {"smoke", "smoky"},\n        "smoky": {"smoke", "smoky"},\n    }.get(normalized, {normalized} if normalized else set())\n\n\ndef split_name_brand_hint(hint: str) -> tuple[str, str | None]:\n    normalized = normalize_text(hint)\n    if " by " not in normalized:\n        return normalized, None\n    name, brand = normalized.rsplit(" by ", 1)\n    return (name.strip(), brand.strip()) if name.strip() and brand.strip() else (normalized, None)\n\n\ndef confidence_adjusted_rating(rating: float, popularity: int, prior: float = 3.8) -> float:\n    if rating <= 0:\n        return 0.0\n    weight = popularity / (popularity + 250) if popularity > 0 else 0.0\n    return rating * weight + prior * (1.0 - weight)\n\n\ndef community_similarity_score(up_votes: int, down_votes: int) -> float:\n    total = up_votes + down_votes\n    if total <= 0:\n        return 0.0\n    approval = (up_votes + 1) / (total + 2)\n    confidence = min(math.log1p(total) / math.log(501), 1.0)\n    return approval * 0.72 + confidence * 0.28\n\n\ndef set_similarity(left: set[str], right: set[str]) -> float:\n    if not left or not right:\n        return 0.0\n    intersection = len(left & right)\n    jaccard = intersection / len(left | right)\n    overlap = intersection / min(len(left), len(right))\n    return jaccard * 0.65 + overlap * 0.35\n\n\ndef trait_match_strength(wanted: str, trait: str) -> float:\n    """Score canonical trait matches without rewarding accidental substrings."""\n    wanted_norm = normalize_text(wanted)\n    trait_norm = normalize_text(trait)\n    if not wanted_norm or not trait_norm:\n        return 0.0\n    if wanted_norm == trait_norm:\n        return 1.0\n    if trait_norm in trait_variants(wanted_norm):\n        return 1.0\n\n    if wanted_norm in {"spicy", "floral"} and wanted_norm in trait_norm.split():\n        return 0.75\n\n    # A multi-word planner term can be a useful refinement of a stored trait,\n    # but a single token such as "fresh" must not match "fresh spicy".\n    if len(wanted_norm.split()) > 1:\n        padded_wanted = f" {wanted_norm} "\n        padded_trait = f" {trait_norm} "\n        if padded_wanted in padded_trait or padded_trait in padded_wanted:\n            return 0.65\n    return 0.0\n\n\ndef metadata_has_term(metadata: dict[str, Any], term: str) -> bool:\n    term_norm = normalize_text(term)\n    if not term_norm:\n        return False\n    searchable = normalize_text(" ".join(\n        str(metadata.get(key) or "")\n        for key in ("name", "brand", "accords_csv", "notes_csv")\n    ))\n    if f" {term_norm} " in f" {searchable} ":\n        return True\n    traits = csv_terms(metadata.get("accords_csv")) | csv_terms(metadata.get("notes_csv"))\n    return any(trait_match_strength(term_norm, trait) > 0.0 for trait in traits)\n\n\nclass CatalogResolver:\n    def __init__(self, catalog_path: Path | str) -> None:\n        self.path = Path(catalog_path)\n        if not self.path.exists():\n            raise FileNotFoundError(f"Catalog does not exist: {self.path}")\n        self.connection = sqlite3.connect(\n            f"file:{self.path}?mode=ro",\n            uri=True,\n            check_same_thread=False,\n        )\n        self.connection.row_factory = sqlite3.Row\n        self.lock = threading.RLock()\n        self._brand_norm_to_name, self._brand_alias_to_norm = self._build_brand_alias_index()\n\n    def _build_brand_alias_index(self) -> tuple[dict[str, str], dict[str, str]]:\n        with self.lock:\n            rows = self.connection.execute(\n                """\n                SELECT brand, SUM(popularity) AS total_popularity\n                FROM perfumes\n                WHERE brand != \'\'\n                GROUP BY brand\n                ORDER BY total_popularity DESC\n                """\n            ).fetchall()\n\n        norm_to_name: dict[str, str] = {}\n        alias_targets: dict[str, set[str]] = defaultdict(set)\n        for row in rows:\n            brand = str(row["brand"] or "").strip()\n            brand_norm = normalize_text(brand)\n            if not brand_norm:\n                continue\n            norm_to_name.setdefault(brand_norm, brand)\n            alias_targets[brand_norm].add(brand_norm)\n            words = brand_norm.split()\n            if 2 <= len(words) <= 5:\n                initials = "".join(word[0] for word in words)\n                spaced_initials = " ".join(word[0] for word in words)\n                if len(initials) >= 2:\n                    alias_targets[initials].add(brand_norm)\n                    alias_targets[spaced_initials].add(brand_norm)\n\n        # Ambiguous initials are intentionally discarded. Guessing a brand is\n        # worse than falling through to the lexical resolver.\n        aliases = {\n            alias: next(iter(targets))\n            for alias, targets in alias_targets.items()\n            if len(targets) == 1\n        }\n        # A compact curated list overrides ambiguous generated initials only\n        # where the fragrance community has a stable, well-known shorthand.\n        for alias, target in EXPLICIT_BRAND_ALIASES.items():\n            if target in norm_to_name:\n                aliases[normalize_text(alias)] = target\n        return norm_to_name, aliases\n\n    def _canonical_brand_norm(self, hint: Any) -> str | None:\n        normalized = normalize_text(hint)\n        return self._brand_alias_to_norm.get(normalized)\n\n    def _extract_edge_brand(self, normalized_hint: str) -> tuple[str, str | None]:\n        tokens = normalized_hint.split()\n        if not tokens:\n            return normalized_hint, None\n        max_width = min(5, len(tokens))\n        matches: list[tuple[int, str, str]] = []\n        for width in range(1, max_width + 1):\n            prefix = " ".join(tokens[:width])\n            suffix = " ".join(tokens[-width:])\n            if prefix in self._brand_alias_to_norm:\n                remainder = tokens[width:]\n                if remainder[:1] == ["s"]:\n                    remainder = remainder[1:]\n                matches.append((width, " ".join(remainder), self._brand_alias_to_norm[prefix]))\n            if suffix in self._brand_alias_to_norm:\n                matches.append((width, " ".join(tokens[:-width]), self._brand_alias_to_norm[suffix]))\n        if not matches:\n            return normalized_hint, None\n        _, name_hint, brand_norm = max(matches, key=lambda item: (item[0], len(item[1])))\n        return name_hint.strip(), brand_norm\n\n    def _prepare_hint(self, hint: str) -> tuple[str, str | None, str]:\n        normalized = expand_concentration_aliases(hint)\n        name_hint, raw_brand_hint = split_name_brand_hint(normalized)\n        if raw_brand_hint:\n            brand_hint = self._canonical_brand_norm(raw_brand_hint)\n            if brand_hint:\n                return name_hint, brand_hint, f"{name_hint} {brand_hint}".strip()\n        edge_name, edge_brand = self._extract_edge_brand(normalized)\n        if edge_brand:\n            return edge_name, edge_brand, f"{edge_name} {edge_brand}".strip()\n        return normalized, None, normalized\n\n    def counts(self) -> dict[str, int]:\n        with self.lock:\n            perfume_count = int(self.connection.execute("SELECT COUNT(*) FROM perfumes").fetchone()[0])\n            edge_count = int(self.connection.execute("SELECT COUNT(*) FROM similarity_edges").fetchone()[0])\n        return {"perfumes": perfume_count, "similarity_edges": edge_count}\n\n    @lru_cache(maxsize=4096)\n    def canonical_brand(self, hint: str) -> str | None:\n        normalized = self._canonical_brand_norm(hint) or normalize_text(hint)\n        if not normalized:\n            return None\n        with self.lock:\n            row = self.connection.execute(\n                """\n                SELECT brand, SUM(popularity) AS total_popularity\n                FROM perfumes\n                WHERE brand_norm = ?\n                GROUP BY brand\n                ORDER BY total_popularity DESC\n                LIMIT 1\n                """,\n                (normalized,),\n            ).fetchone()\n        return str(row["brand"]) if row else None\n\n    @lru_cache(maxsize=4096)\n    def resolve(self, hint: str) -> dict[str, Any] | None:\n        normalized = expand_concentration_aliases(hint)\n        if not normalized:\n            return None\n        name_hint, brand_hint, label_hint = self._prepare_hint(hint)\n\n        with self.lock:\n            if brand_hint:\n                exact = self.connection.execute(\n                    """\n                    SELECT * FROM perfumes\n                    WHERE (name_norm = ? AND brand_norm = ?) OR label_norm = ?\n                    ORDER BY popularity DESC LIMIT 20\n                    """,\n                    (name_hint, brand_hint, label_hint),\n                ).fetchall()\n            else:\n                exact = self.connection.execute(\n                    """\n                    SELECT * FROM perfumes\n                    WHERE name_norm = ? OR label_norm = ?\n                    ORDER BY popularity DESC LIMIT 20\n                    """,\n                    (name_hint, label_hint),\n                ).fetchall()\n        if exact:\n            return dict(exact[0])\n\n        family_found, family = self._dominant_family_member(name_hint, brand_hint)\n        if family_found:\n            return family\n\n        tokens = [token for token in name_hint.split() if len(token) >= 2]\n        if not tokens and brand_hint and name_hint:\n            tokens = [name_hint]\n        if not tokens:\n            return None\n        clauses: list[str] = []\n        values: list[str] = []\n        for token in tokens[:5]:\n            clauses.append("(name_norm LIKE ? OR brand_norm LIKE ?)")\n            values.extend([f"%{token}%", f"%{token}%"])\n        brand_clause = " AND brand_norm = ?" if brand_hint else ""\n        if brand_hint:\n            values.append(brand_hint)\n        with self.lock:\n            rows = self.connection.execute(\n                "SELECT * FROM perfumes WHERE "\n                + "(" + " OR ".join(clauses) + ")"\n                + brand_clause\n                + " ORDER BY popularity DESC LIMIT 500",\n                values,\n            ).fetchall()\n            if not rows and brand_hint:\n                # When the brand is certain, a misspelled short product name\n                # should still be compared against that house\'s catalog.\n                rows = self.connection.execute(\n                    """\n                    SELECT * FROM perfumes\n                    WHERE brand_norm = ?\n                    ORDER BY popularity DESC, rating DESC\n                    LIMIT 500\n                    """,\n                    (brand_hint,),\n                ).fetchall()\n        ranked = sorted(\n            (dict(row) for row in rows),\n            key=lambda row: self._resolution_score(name_hint, brand_hint, row),\n            reverse=True,\n        )\n        if not ranked or self._resolution_score(name_hint, brand_hint, ranked[0]) < 0.60:\n            return None\n        return ranked[0]\n\n    def _dominant_family_member(\n        self,\n        name_hint: str,\n        brand_hint: str | None,\n    ) -> tuple[bool, dict[str, Any] | None]:\n        if len(name_hint.split()) < 2:\n            return False, None\n        values: list[Any] = [f"{name_hint} %"]\n        brand_clause = ""\n        if brand_hint:\n            brand_clause = " AND brand_norm = ?"\n            values.append(brand_hint)\n        with self.lock:\n            rows = self.connection.execute(\n                "SELECT * FROM perfumes WHERE name_norm LIKE ?"\n                + brand_clause\n                + " ORDER BY popularity DESC, rating DESC LIMIT 25",\n                values,\n            ).fetchall()\n        if not rows:\n            return False, None\n        ranked = [dict(row) for row in rows]\n        leader = ranked[0]\n        leader_popularity = int(leader.get("popularity") or 0)\n        runner_up = int(ranked[1].get("popularity") or 0) if len(ranked) > 1 else 0\n        clearly_dominant = (\n            leader_popularity >= 500\n            and (runner_up == 0 or leader_popularity >= runner_up * 2)\n        )\n        return True, leader if clearly_dominant else None\n\n    @staticmethod\n    def _resolution_score(\n        name_hint: str,\n        brand_hint: str | None,\n        row: dict[str, Any],\n    ) -> float:\n        name = str(row.get("name_norm") or "")\n        row_brand = str(row.get("brand_norm") or "")\n        hint_core, hint_qualifiers = identity_signature(name_hint)\n        name_core, name_qualifiers = identity_signature(name)\n        hint_core_text = " ".join(hint_core)\n        name_core_text = " ".join(name_core)\n        hint_tokens = set(hint_core)\n        name_tokens = set(name_core)\n        overlap = len(hint_tokens & name_tokens) / max(len(hint_tokens | name_tokens), 1)\n        coverage = len(hint_tokens & name_tokens) / max(len(hint_tokens), 1)\n        core_sequence = difflib.SequenceMatcher(None, hint_core_text, name_core_text).ratio()\n        full_sequence = difflib.SequenceMatcher(\n            None,\n            expand_concentration_aliases(name_hint),\n            name,\n        ).ratio()\n        core_lexical = max(core_sequence, overlap, coverage)\n        # Edition words such as "eau de toilette" are common across thousands\n        # of unrelated perfumes. The product-name core must dominate them.\n        lexical = core_lexical * 0.90 + full_sequence * 0.10\n        core_mismatch_penalty = (\n            0.38\n            if hint_tokens and name_tokens and not (hint_tokens & name_tokens) and core_sequence < 0.72\n            else 0.0\n        )\n\n        qualifier_score = 0.5\n        qualifier_penalty = 0.0\n        if hint_qualifiers:\n            qualifier_score = len(hint_qualifiers & name_qualifiers) / len(hint_qualifiers)\n            # Some catalog base releases omit EDT/EDC from their display name.\n            # Missing edition text is a moderate penalty; an explicitly\n            # conflicting concentration remains a hard penalty below.\n            qualifier_penalty += 0.06 * (\n                len(hint_qualifiers - name_qualifiers) / len(hint_qualifiers)\n            )\n            requested_concentration = hint_qualifiers & CONCENTRATION_TOKENS\n            candidate_concentration = name_qualifiers & CONCENTRATION_TOKENS\n            if requested_concentration and candidate_concentration and not (\n                requested_concentration & candidate_concentration\n            ):\n                qualifier_penalty += 0.32\n\n        brand_score = 1.0 if brand_hint and row_brand == brand_hint else (0.5 if not brand_hint else 0.0)\n        popularity = int(row.get("popularity") or 0)\n        popularity_score = min(math.log10(popularity + 1) / math.log10(50000), 1.0) if popularity else 0.0\n        return (\n            lexical * 0.72\n            + qualifier_score * core_lexical * 0.18\n            + brand_score * 0.07\n            + popularity_score * 0.03\n            - qualifier_penalty\n            - core_mismatch_penalty\n        )\n\n    def direct_similarity(self, source_id: int, limit: int = 200) -> list[dict[str, Any]]:\n        with self.lock:\n            rows = self.connection.execute(\n                """\n                SELECT p.*, e.up_votes, e.down_votes\n                FROM similarity_edges e\n                JOIN perfumes p ON p.perfume_id = e.target_id\n                WHERE e.source_id = ?\n                ORDER BY (e.up_votes - e.down_votes) DESC, e.up_votes DESC\n                LIMIT ?\n                """,\n                (source_id, limit),\n            ).fetchall()\n        return [dict(row) for row in rows]\n\n    def popular_candidates(\n        self,\n        filters: dict[str, Any],\n        required_terms: list[str],\n        exclude_terms: list[str],\n        *,\n        canonical_brand: str | None = None,\n        limit: int = 160,\n    ) -> list[dict[str, Any]]:\n        """Return popular eligible rows independently of the ANN neighborhood."""\n        clauses: list[str] = []\n        values: list[Any] = []\n\n        gender = normalize_text(filters.get("gender"))\n        if gender:\n            allowed = [gender] if gender == "unisex" else [gender, "unisex"]\n            clauses.append("gender IN (" + ", ".join("?" for _ in allowed) + ")")\n            values.extend(allowed)\n        if canonical_brand:\n            clauses.append("brand = ?")\n            values.append(canonical_brand)\n        if filters.get("min_rating") is not None:\n            clauses.append("rating >= ?")\n            values.append(float(filters["min_rating"]))\n        if filters.get("min_popularity") is not None:\n            clauses.append("popularity >= ?")\n            values.append(int(filters["min_popularity"]))\n\n        sql = "SELECT * FROM perfumes"\n        if clauses:\n            sql += " WHERE " + " AND ".join(clauses)\n        # Scan a generous popularity-first window, then apply taxonomy-aware\n        # trait and season/time checks in Python.\n        sql += " ORDER BY popularity DESC, rating DESC LIMIT 4000"\n        with self.lock:\n            rows = [dict(row) for row in self.connection.execute(sql, values).fetchall()]\n\n        season = normalize_text(filters.get("season"))\n        time_profile = normalize_text(filters.get("time"))\n        output: list[dict[str, Any]] = []\n        for row in rows:\n            if season and season not in csv_terms(row.get("seasons_csv")):\n                continue\n            if time_profile and time_profile not in csv_terms(row.get("time_profile_csv")):\n                continue\n            traits = csv_terms(row.get("accords_csv")) | csv_terms(row.get("notes_csv"))\n            if any(\n                not any(trait_match_strength(term, trait) > 0.0 for trait in traits)\n                for term in required_terms\n            ):\n                continue\n            if any(metadata_has_term(row, term) for term in exclude_terms):\n                continue\n            output.append(row)\n            if len(output) >= limit:\n                break\n        return output\n\n\nclass RetrievalEngine:\n    def __init__(\n        self,\n        db_dir: Path | str,\n        catalog_path: Path | str,\n        *,\n        collection_name: str = DEFAULT_COLLECTION,\n        model_name: str = DEFAULT_MODEL,\n    ) -> None:\n        import chromadb\n        from sentence_transformers import SentenceTransformer\n\n        self.started_at = time.time()\n        self.db_dir = Path(db_dir)\n        self.collection_name = collection_name\n        self.model_name = model_name\n        self.catalog = CatalogResolver(catalog_path)\n        self.client = chromadb.PersistentClient(path=str(self.db_dir))\n        self.collection = self.client.get_collection(collection_name)\n        self.model = SentenceTransformer(model_name, device="cpu")\n        self.encode_lock = threading.Lock()\n\n    def health(self) -> dict[str, Any]:\n        return {\n            "status": "ok",\n            "model": self.model_name,\n            "device": "cpu",\n            "collection": self.collection_name,\n            "collection_count": self.collection.count(),\n            "catalog": self.catalog.counts(),\n            "embedding_cache": self._encode.cache_info()._asdict(),\n            "uptime_seconds": round(time.time() - self.started_at, 2),\n        }\n\n    @lru_cache(maxsize=512)\n    def _encode(self, text: str) -> tuple[float, ...]:\n        with self.encode_lock:\n            vector = self.model.encode(text, normalize_embeddings=True)\n        return tuple(float(value) for value in vector)\n\n    def search(self, payload: dict[str, Any]) -> dict[str, Any]:\n        started = time.perf_counter()\n        query = str(payload.get("query") or "").strip()\n        if not query:\n            raise ValueError("query is required")\n        top_k = min(max(int(payload.get("top_k") or 10), 1), 30)\n        fetch_k = min(max(int(payload.get("fetch_k") or 300), top_k), 300)\n        filters = dict(payload.get("filters") or {})\n        exclude_terms = self._clean_terms(payload.get("exclude_terms") or [])\n        wanted_terms = self._clean_terms(payload.get("wanted_terms") or [])\n        required_terms = self._clean_terms(payload.get("required_terms") or [])\n        exclude_ids = self._clean_ids(payload.get("exclude_ids") or [])\n        discovery_mode = normalize_text(payload.get("discovery_mode") or "balanced")\n        if discovery_mode not in VALID_DISCOVERY_MODES:\n            raise ValueError(f"Unsupported discovery mode: {discovery_mode}")\n        where, canonical_brand = self._build_where(filters)\n        normalized_query = normalize_text(query)\n        padded_query = f" {normalized_query} "\n\n        encoded_query = BGE_QUERY_PREFIX + query\n        before = self._encode.cache_info().hits\n        embedding = [list(self._encode(encoded_query))]\n        cache_hit = self._encode.cache_info().hits > before\n        raw = self.collection.query(\n            query_embeddings=embedding,\n            n_results=fetch_k,\n            where=where,\n            include=["documents", "metadatas", "distances"],\n        )\n\n        raw_rows = [(*row, {"semantic_ann"}) for row in zip(\n            raw["documents"][0],\n            raw["metadatas"][0],\n            raw["distances"][0],\n        )]\n\n        # ANN relevance and catalog popularity are complementary candidate\n        # generators. This branch prevents famous, eligible perfumes from\n        # disappearing merely because their names/cards are not close enough\n        # to a short lifestyle query in the first ANN neighborhood.\n        if discovery_mode != "niche":\n            popular_rows = self.catalog.popular_candidates(\n                filters,\n                required_terms,\n                exclude_terms,\n                canonical_brand=canonical_brand,\n                limit=200 if discovery_mode == "mainstream" else 140,\n            )\n            popular_ids = [str(row["perfume_id"]) for row in popular_rows]\n            if popular_ids:\n                popular = self.collection.get(\n                    ids=popular_ids,\n                    include=["documents", "metadatas", "embeddings"],\n                )\n                existing = {int(metadata.get("perfume_id") or 0): index for index, (_, metadata, _, _) in enumerate(raw_rows)}\n                for document, metadata, vector in zip(\n                    popular.get("documents") or [],\n                    popular.get("metadatas") or [],\n                    popular.get("embeddings") if popular.get("embeddings") is not None else [],\n                ):\n                    perfume_id = int(metadata.get("perfume_id") or 0)\n                    if perfume_id in existing:\n                        raw_rows[existing[perfume_id]][3].add("catalog_popular")\n                        continue\n                    semantic = sum(float(left) * float(right) for left, right in zip(embedding[0], vector))\n                    raw_rows.append((document, metadata, 1.0 - semantic, {"catalog_popular"}))\n        fetched_trait_sets = [\n            csv_terms(metadata.get("accords_csv")) | csv_terms(metadata.get("notes_csv"))\n            for _, metadata, _, _ in raw_rows\n        ]\n        supported_wanted_terms = [\n            term\n            for term in wanted_terms\n            if any(\n                any(trait_match_strength(term, trait) > 0.0 for trait in traits)\n                for traits in fetched_trait_sets\n            )\n        ]\n        supported_required_terms = [\n            term\n            for term in required_terms\n            if any(\n                any(trait_match_strength(term, trait) > 0.0 for trait in traits)\n                for traits in fetched_trait_sets\n            )\n        ]\n\n        candidates: list[dict[str, Any]] = []\n        collision_brands: set[str] = set()\n        score_weights = {\n            # Balanced keeps semantic fit as the largest signal. Popularity\n            # earns a candidate entry into the race, not an automatic win.\n            "balanced": (0.62, 0.16, 0.08, 0.14),\n            "mainstream": (0.48, 0.29, 0.09, 0.14),\n            "niche": (0.70, 0.05, 0.11, 0.14),\n        }[discovery_mode]\n        semantic_weight, popularity_weight, rating_weight, wanted_weight = score_weights\n        for document, metadata, distance, source_pools in raw_rows:\n            meta = dict(metadata)\n            perfume_id = int(meta.get("perfume_id") or 0)\n            if perfume_id in exclude_ids:\n                continue\n            if self._has_excluded_term(meta, exclude_terms):\n                continue\n            semantic = max(0.0, 1.0 - float(distance))\n            popularity = int(meta.get("popularity") or 0)\n            rating = confidence_adjusted_rating(float(meta.get("rating") or 0.0), popularity)\n            popularity_score = min(math.log10(popularity + 1) / math.log10(50000), 1.0) if popularity else 0.0\n            rating_score = min(max((rating - 3.2) / 1.3, 0.0), 1.0) if rating else 0.0\n            traits = csv_terms(meta.get("accords_csv")) | csv_terms(meta.get("notes_csv"))\n            if required_terms and (\n                len(supported_required_terms) != len(required_terms)\n                or any(\n                    not any(trait_match_strength(term, trait) > 0.0 for trait in traits)\n                    for term in supported_required_terms\n                )\n            ):\n                continue\n            wanted_matches = {\n                term: max((trait_match_strength(term, trait) for trait in traits), default=0.0)\n                for term in supported_wanted_terms\n            }\n            wanted_matches = {\n                term: strength for term, strength in wanted_matches.items() if strength > 0.0\n            }\n            wanted_hits = len(wanted_matches)\n            wanted_score = (\n                sum(wanted_matches.values()) / len(supported_wanted_terms)\n                if supported_wanted_terms\n                else 0.0\n            )\n            brand_norm = normalize_text(meta.get("brand"))\n            brand_collision = bool(\n                not canonical_brand\n                and len(brand_norm) >= 4\n                and f" {brand_norm} " in padded_query\n            )\n            if brand_collision:\n                collision_brands.add(brand_norm)\n            collision_penalty = 0.14 if brand_collision else 0.0\n            final_score = (\n                semantic * semantic_weight\n                + popularity_score * popularity_weight\n                + rating_score * rating_weight\n                + wanted_score * wanted_weight\n                - collision_penalty\n            )\n            candidates.append(\n                {\n                    "perfume_id": perfume_id,\n                    "name": str(meta.get("name") or ""),\n                    "brand": str(meta.get("brand") or ""),\n                    "label": perfume_label(meta.get("name"), meta.get("brand")),\n                    "document": document,\n                    "metadata": meta,\n                    "distance": round(float(distance), 6),\n                    "semantic_score": round(semantic, 6),\n                    "score": round(final_score, 6),\n                    "reasons": {\n                        "semantic": round(semantic, 4),\n                        "popularity": round(popularity_score, 4),\n                        "rating": round(rating_score, 4),\n                        "wanted_term_hits": wanted_hits,\n                        "wanted_term_score": round(wanted_score, 4),\n                        "wanted_term_matches": {\n                            term: round(strength, 2)\n                            for term, strength in wanted_matches.items()\n                        },\n                        "accidental_brand_collision_penalty": collision_penalty,\n                        "candidate_sources": sorted(source_pools),\n                    },\n                }\n            )\n        candidates.sort(key=lambda item: item["score"], reverse=True)\n        selected = self._diversify(\n            candidates,\n            top_k,\n            brand_limit=top_k if canonical_brand else 2,\n            per_brand_limits={brand: 1 for brand in collision_brands},\n        )\n        return {\n            "route": "semantic",\n            "query": query,\n            "filters": {**filters, "canonical_brand": canonical_brand},\n            "exclude_terms": exclude_terms,\n            "wanted_terms": wanted_terms,\n            "supported_wanted_terms": supported_wanted_terms,\n            "ignored_wanted_terms": [term for term in wanted_terms if term not in supported_wanted_terms],\n            "required_terms": required_terms,\n            "supported_required_terms": supported_required_terms,\n            "exclude_ids": sorted(exclude_ids),\n            "discovery_mode": discovery_mode,\n            "accidental_brand_collisions": sorted(collision_brands),\n            "embedding_cache_hit": cache_hit,\n            "elapsed_seconds": round(time.perf_counter() - started, 4),\n            "result_count": len(selected),\n            "results": selected,\n        }\n\n    def resolve(self, payload: dict[str, Any]) -> dict[str, Any]:\n        hint = str(payload.get("hint") or "").strip()\n        if not hint:\n            raise ValueError("hint is required")\n        row = self.catalog.resolve(hint)\n        return {"hint": hint, "resolved": self._public_catalog_row(row) if row else None}\n\n    def similar(self, payload: dict[str, Any]) -> dict[str, Any]:\n        started = time.perf_counter()\n        hint = str(payload.get("hint") or "").strip()\n        if not hint:\n            raise ValueError("hint is required")\n        source = self.catalog.resolve(hint)\n        if not source:\n            raise LookupError(f"Could not resolve perfume: {hint}")\n        top_k = min(max(int(payload.get("top_k") or 10), 1), 30)\n        exclude_terms = self._clean_terms(payload.get("exclude_terms") or [])\n        required_terms = self._clean_terms(payload.get("required_terms") or [])\n        exclude_ids = self._clean_ids(payload.get("exclude_ids") or [])\n        exclude_source_brand = bool(payload.get("exclude_source_brand", False))\n        source_accords = csv_terms(source.get("accords_csv"))\n        source_notes = csv_terms(source.get("notes_csv"))\n\n        scored: list[dict[str, Any]] = []\n        for row in self.catalog.direct_similarity(int(source["perfume_id"]), limit=max(200, top_k * 15)):\n            if int(row.get("perfume_id") or 0) in exclude_ids:\n                continue\n            if exclude_source_brand and normalize_text(row.get("brand")) == normalize_text(source.get("brand")):\n                continue\n            if self._has_excluded_term(row, exclude_terms):\n                continue\n            row_traits = csv_terms(row.get("accords_csv")) | csv_terms(row.get("notes_csv"))\n            if any(\n                not any(trait_match_strength(term, trait) > 0.0 for trait in row_traits)\n                for term in required_terms\n            ):\n                continue\n            graph = community_similarity_score(int(row.get("up_votes") or 0), int(row.get("down_votes") or 0))\n            accord_similarity = set_similarity(source_accords, csv_terms(row.get("accords_csv")))\n            note_similarity = set_similarity(source_notes, csv_terms(row.get("notes_csv")))\n            structure = accord_similarity * 0.68 + note_similarity * 0.32\n            popularity = int(row.get("popularity") or 0)\n            quality = min(math.log10(popularity + 1) / math.log10(50000), 1.0) if popularity else 0.0\n            final_score = graph * 0.68 + structure * 0.24 + quality * 0.08\n            public = self._public_catalog_row(row)\n            assert public is not None\n            public.update(\n                {\n                    "score": round(final_score, 6),\n                    "community_similarity": round(graph, 6),\n                    "accord_similarity": round(accord_similarity, 6),\n                    "note_similarity": round(note_similarity, 6),\n                    "up_votes": int(row.get("up_votes") or 0),\n                    "down_votes": int(row.get("down_votes") or 0),\n                }\n            )\n            scored.append(public)\n        scored.sort(key=lambda item: item["score"], reverse=True)\n        selected = self._diversify(scored, top_k, brand_limit=2)\n        return {\n            "route": "community_similarity",\n            "hint": hint,\n            "source": self._public_catalog_row(source),\n            "exclude_terms": exclude_terms,\n            "required_terms": required_terms,\n            "exclude_ids": sorted(exclude_ids),\n            "elapsed_seconds": round(time.perf_counter() - started, 4),\n            "result_count": len(selected),\n            "results": selected,\n        }\n\n    def _build_where(self, filters: dict[str, Any]) -> tuple[dict[str, Any] | None, str | None]:\n        clauses: list[dict[str, Any]] = []\n        gender = normalize_text(filters.get("gender"))\n        if gender:\n            if gender not in VALID_GENDERS:\n                raise ValueError(f"Unsupported gender filter: {gender}")\n            allowed = [gender] if gender == "unisex" else [gender, "unisex"]\n            clauses.append({"gender": {"$in": allowed}})\n        season = normalize_text(filters.get("season"))\n        if season:\n            if season not in VALID_SEASONS:\n                raise ValueError(f"Unsupported season filter: {season}")\n            clauses.append({f"season_{season}": True})\n        time_profile = normalize_text(filters.get("time"))\n        if time_profile:\n            if time_profile not in VALID_TIMES:\n                raise ValueError(f"Unsupported time filter: {time_profile}")\n            clauses.append({f"time_{time_profile}": True})\n        if filters.get("min_rating") is not None:\n            clauses.append({"rating": {"$gte": float(filters["min_rating"])}})\n        if filters.get("min_popularity") is not None:\n            clauses.append({"popularity": {"$gte": int(filters["min_popularity"])}})\n\n        canonical_brand = None\n        brand_hint = str(filters.get("brand") or "").strip()\n        if brand_hint:\n            canonical_brand = self.catalog.canonical_brand(brand_hint)\n            if not canonical_brand:\n                raise LookupError(f"Unknown brand: {brand_hint}")\n            clauses.append({"brand": canonical_brand})\n\n        if not clauses:\n            return None, canonical_brand\n        if len(clauses) == 1:\n            return clauses[0], canonical_brand\n        return {"$and": clauses}, canonical_brand\n\n    @staticmethod\n    def _clean_terms(values: Any) -> list[str]:\n        if not isinstance(values, list):\n            raise ValueError("wanted_terms, required_terms, and exclude_terms must be JSON arrays")\n        return list(dict.fromkeys(term for value in values if (term := normalize_text(value))))[:30]\n\n    @staticmethod\n    def _clean_ids(values: Any) -> set[int]:\n        if not isinstance(values, list):\n            raise ValueError("exclude_ids must be a JSON array")\n        output: set[int] = set()\n        for value in values[:200]:\n            try:\n                perfume_id = int(value)\n            except (TypeError, ValueError):\n                continue\n            if perfume_id > 0:\n                output.add(perfume_id)\n        return output\n\n    @staticmethod\n    def _has_excluded_term(metadata: dict[str, Any], excluded: list[str]) -> bool:\n        return any(metadata_has_term(metadata, term) for term in excluded)\n\n    @staticmethod\n    def _diversify(\n        items: list[dict[str, Any]],\n        top_k: int,\n        *,\n        brand_limit: int,\n        per_brand_limits: dict[str, int] | None = None,\n    ) -> list[dict[str, Any]]:\n        output: list[dict[str, Any]] = []\n        seen_ids: set[int] = set()\n        seen_labels: set[str] = set()\n        brand_counts: dict[str, int] = {}\n        normalized_limits = {\n            normalize_text(brand): int(limit)\n            for brand, limit in (per_brand_limits or {}).items()\n        }\n        for item in items:\n            perfume_id = int(item.get("perfume_id") or 0)\n            label = normalize_text(perfume_label(item.get("name"), item.get("brand")))\n            brand = normalize_text(item.get("brand"))\n            current_limit = normalized_limits.get(brand, brand_limit)\n            if (\n                perfume_id in seen_ids\n                or label in seen_labels\n                or brand_counts.get(brand, 0) >= current_limit\n            ):\n                continue\n            output.append(item)\n            seen_ids.add(perfume_id)\n            seen_labels.add(label)\n            brand_counts[brand] = brand_counts.get(brand, 0) + 1\n            if len(output) >= top_k:\n                break\n        return output\n\n    @staticmethod\n    def _public_catalog_row(row: dict[str, Any] | None) -> dict[str, Any] | None:\n        if not row:\n            return None\n        return {\n            "perfume_id": int(row["perfume_id"]),\n            "name": str(row.get("name") or ""),\n            "brand": str(row.get("brand") or ""),\n            "label": perfume_label(row.get("name"), row.get("brand")),\n            "gender": str(row.get("gender") or ""),\n            "year": row.get("year"),\n            "rating": row.get("rating"),\n            "popularity": int(row.get("popularity") or 0),\n            "longevity": row.get("longevity"),\n            "sillage": row.get("sillage"),\n            "value_score": row.get("value_score"),\n            "accords_csv": str(row.get("accords_csv") or ""),\n            "notes_csv": str(row.get("notes_csv") or ""),\n            "seasons_csv": str(row.get("seasons_csv") or ""),\n            "time_profile_csv": str(row.get("time_profile_csv") or ""),\n        }\n\n\nclass RetrievalRequestHandler(BaseHTTPRequestHandler):\n    engine: RetrievalEngine\n    server_version = "ScentAIRetrieval/1.0"\n\n    def do_GET(self) -> None:  # noqa: N802\n        if self.path == "/health":\n            self._send_json(HTTPStatus.OK, self.engine.health())\n            return\n        self._send_json(HTTPStatus.NOT_FOUND, {"error": "not_found"})\n\n    def do_POST(self) -> None:  # noqa: N802\n        try:\n            payload = self._read_json()\n            if self.path == "/search":\n                output = self.engine.search(payload)\n            elif self.path == "/resolve":\n                output = self.engine.resolve(payload)\n            elif self.path == "/similar":\n                output = self.engine.similar(payload)\n            else:\n                self._send_json(HTTPStatus.NOT_FOUND, {"error": "not_found"})\n                return\n            self._send_json(HTTPStatus.OK, output)\n        except LookupError as exc:\n            self._send_json(HTTPStatus.NOT_FOUND, {"error": str(exc)})\n        except (TypeError, ValueError) as exc:\n            self._send_json(HTTPStatus.BAD_REQUEST, {"error": str(exc)})\n        except Exception as exc:\n            self._send_json(HTTPStatus.INTERNAL_SERVER_ERROR, {"error": repr(exc)})\n\n    def _read_json(self) -> dict[str, Any]:\n        length = int(self.headers.get("Content-Length") or 0)\n        if length <= 0 or length > 1_000_000:\n            raise ValueError("Request body must contain at most 1 MB of JSON")\n        payload = json.loads(self.rfile.read(length).decode("utf-8"))\n        if not isinstance(payload, dict):\n            raise ValueError("JSON request must be an object")\n        return payload\n\n    def _send_json(self, status: HTTPStatus, payload: dict[str, Any]) -> None:\n        encoded = json.dumps(payload, ensure_ascii=False, allow_nan=False).encode("utf-8")\n        self.send_response(status.value)\n        self.send_header("Content-Type", "application/json; charset=utf-8")\n        self.send_header("Content-Length", str(len(encoded)))\n        self.end_headers()\n        self.wfile.write(encoded)\n\n    def log_message(self, format: str, *args: Any) -> None:\n        print(f"[retrieval-http] {self.address_string()} {format % args}", flush=True)\n\n\ndef main() -> None:\n    db_dir = Path(os.environ["SCENTAI_CHROMA_DIR"])\n    catalog_path = Path(os.environ["SCENTAI_CATALOG_PATH"])\n    host = os.environ.get("SCENTAI_RETRIEVAL_HOST", "127.0.0.1")\n    port = int(os.environ.get("SCENTAI_RETRIEVAL_PORT", "8020"))\n    engine = RetrievalEngine(db_dir, catalog_path)\n    RetrievalRequestHandler.engine = engine\n    server = ThreadingHTTPServer((host, port), RetrievalRequestHandler)\n    print(json.dumps({"event": "ready", "host": host, "port": port, **engine.health()}), flush=True)\n    server.serve_forever()\n\n\nif __name__ == "__main__":\n    main()\n'


import urllib.error
import urllib.request

SERVICE_PATH = Path("/content/scentai_retrieval_service.py")
SERVICE_LOG = Path("/content/scentai_retrieval_service.log")
SERVICE_PATH.write_text(SERVICE_SOURCE, encoding="utf-8")
subprocess.run([str(RETRIEVAL_PYTHON), "-m", "py_compile", str(SERVICE_PATH)], check=True)

RETRIEVAL_HOST = "127.0.0.1"
RETRIEVAL_PORT = 8020
RETRIEVAL_URL = f"http://{RETRIEVAL_HOST}:{RETRIEVAL_PORT}"


def get_json(url, *, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def service_is_healthy():
    try:
        return get_json(f"{RETRIEVAL_URL}/health", timeout=2).get("status") == "ok"
    except (OSError, urllib.error.URLError, json.JSONDecodeError):
        return False


def service_log_tail(line_count=80):
    if not SERVICE_LOG.exists():
        return "<service log has not been created>"
    return "\\n".join(
        SERVICE_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-line_count:]
    )


if service_is_healthy():
    print("Reusing healthy retrieval service.")
else:
    service_environment = os.environ.copy()
    service_environment.update({
        "PYTHONUNBUFFERED": "1",
        "ANONYMIZED_TELEMETRY": "False",
        "TOKENIZERS_PARALLELISM": "false",
        "HF_HOME": "/content/huggingface_retrieval_cache",
        "SCENTAI_CHROMA_DIR": str(LOCAL_CHROMA_DIR),
        "SCENTAI_CATALOG_PATH": str(LOCAL_CATALOG),
        "SCENTAI_RETRIEVAL_HOST": RETRIEVAL_HOST,
        "SCENTAI_RETRIEVAL_PORT": str(RETRIEVAL_PORT),
        "OMP_NUM_THREADS": str(min(os.cpu_count() or 2, 8)),
    })
    if HF_TOKEN:
        service_environment["HF_TOKEN"] = HF_TOKEN
    service_log_handle = SERVICE_LOG.open("w", encoding="utf-8")
    service_process = subprocess.Popen(
        [str(RETRIEVAL_PYTHON), str(SERVICE_PATH)],
        stdout=service_log_handle,
        stderr=subprocess.STDOUT,
        env=service_environment,
    )
    print("Starting BGE-M3 retrieval service. First model download/load can take several minutes.")
    deadline = time.monotonic() + 1800
    next_status = time.monotonic() + 30
    while not service_is_healthy():
        return_code = service_process.poll()
        if return_code is not None:
            service_log_handle.close()
            raise RuntimeError(
                f"Retrieval service exited with code {return_code}. Last log lines:\\n{service_log_tail()}"
            )
        if time.monotonic() >= deadline:
            service_process.terminate()
            service_log_handle.close()
            raise TimeoutError("Retrieval service startup timed out.\\n" + service_log_tail())
        if time.monotonic() >= next_status:
            print("Still loading...\\n" + service_log_tail(10))
            next_status = time.monotonic() + 30
        time.sleep(3)

health = get_json(f"{RETRIEVAL_URL}/health")
print(json.dumps(health, indent=2))
assert health["collection_count"] == 131930, health
assert health["catalog"]["perfumes"] == 131930, health
assert health["catalog"]["similarity_edges"] > 600000, health
print("Retrieval service preflight: passed")

## Contract and quality smoke tests

In [ ]:
def post_json(path, payload, *, timeout=180):
    request = urllib.request.Request(
        f"{RETRIEVAL_URL}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Retrieval HTTP {exc.code}: {body}") from exc


resolved = post_json("/resolve", {"hint": "Club de Nuit by Armaf"})
print("Resolved short name:", resolved["resolved"]["label"])
assert resolved["resolved"]["label"] == "Club de Nuit Intense Man by Armaf"

identity_contracts = {
    "YSL Y EDP": "Y Eau de Parfum by Yves Saint Laurent",
    "Y EPD by YSL": "Y Eau de Parfum by Yves Saint Laurent",
    "Bleu de Chanel EDP": "Bleu de Chanel Eau de Parfum by Chanel",
    "Bleu de Chanel EDT": "Bleu de Chanel by Chanel",
    "JPG Le Male Le Parfum": "Le Male Le Parfum by Jean Paul Gaultier",
    "LV Imagination": "Imagination by Louis Vuitton",
    "MFK Grand Soir": "Grand Soir by Maison Francis Kurkdjian",
    "PDM Layton": "Layton by Parfums de Marly",
}
for hint, expected_label in identity_contracts.items():
    identity = post_json("/resolve", {"hint": hint})["resolved"]
    assert identity and identity["label"] == expected_label, {"hint": hint, "resolved": identity}
print("Alias/edition identity contracts:", list(identity_contracts))

versace = post_json(
    "/search",
    {
        "query": "Recommend men's fragrances from Versace",
        "top_k": 8,
        "filters": {"brand": "Versace", "gender": "male"},
    },
)
assert len(versace["results"]) >= 5, versace
assert all(item["brand"] == "Versace" for item in versace["results"]), versace
print("Versace brand filter:", [item["label"] for item in versace["results"][:5]])

office = post_json(
    "/search",
    {
        "query": "I need a clean office scent without vanilla.",
        "top_k": 10,
        "filters": {"time": "day", "min_popularity": 100},
        "wanted_terms": ["clean", "fresh", "soapy", "powdery"],
        "exclude_terms": ["vanilla"],
    },
)
assert office["results"], office
for item in office["results"]:
    traits = (item["metadata"].get("accords_csv", "") + "," + item["metadata"].get("notes_csv", "")).lower()
    assert "vanilla" not in traits, item
assert sum(item["brand"] == "Clean" for item in office["results"]) <= 1, office
print("Office/no-vanilla:", [item["label"] for item in office["results"][:5]])

similar = post_json("/similar", {"hint": "Aventus", "top_k": 10})
assert similar["source"]["label"] == "Aventus by Creed", similar
assert any(item["label"] == "Club de Nuit Intense Man by Armaf" for item in similar["results"][:5]), similar
print("Aventus graph matches:", [item["label"] for item in similar["results"][:5]])

print("\\nCLEAN RETRIEVAL CONTRACT TEST: PASSED")

## Warm latency benchmark and durable report

In [ ]:
import statistics
from datetime import datetime, timezone

benchmark_cases = [
    {"query": "fresh citrus summer cologne for men", "filters": {"gender": "male", "season": "summer"}, "wanted_terms": ["citrus", "fresh"]},
    {"query": "warm spicy fragrance for a winter date night", "filters": {"season": "winter", "time": "night"}, "wanted_terms": ["warm spicy"]},
    {"query": "clean office scent without vanilla", "filters": {"time": "day"}, "exclude_terms": ["vanilla"], "wanted_terms": ["clean", "soapy"]},
    {"query": "pineapple woody fragrance", "wanted_terms": ["pineapple", "woody"]},
    {"query": "old school masculine fougere", "filters": {"gender": "male"}, "wanted_terms": ["fougere", "aromatic"]},
]

rounds = []
for round_number in (1, 2):
    for case in benchmark_cases:
        wall_started = time.perf_counter()
        result = post_json("/search", {**case, "top_k": 10})
        rounds.append({
            "round": round_number,
            "query": case["query"],
            "wall_seconds": round(time.perf_counter() - wall_started, 4),
            "service_seconds": result["elapsed_seconds"],
            "embedding_cache_hit": result["embedding_cache_hit"],
            "top_labels": [item["label"] for item in result["results"][:5]],
        })
        print(round_number, case["query"], rounds[-1]["wall_seconds"], "s", "cache=", result["embedding_cache_hit"])

warm = [row["wall_seconds"] for row in rounds if row["round"] == 2]
assert all(row["embedding_cache_hit"] for row in rounds if row["round"] == 2), rounds
report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "stage": "clean_retrieval_stage2",
    "health": get_json(f"{RETRIEVAL_URL}/health"),
    "copy_elapsed_seconds": round(copy_elapsed, 4),
    "warm_mean_seconds": round(statistics.mean(warm), 4),
    "warm_median_seconds": round(statistics.median(warm), 4),
    "runs": rounds,
    "contract_tests": {
        "canonical_family_resolution": True,
        "alias_and_edition_resolution": True,
        "brand_filter": True,
        "negative_filter": True,
        "community_similarity": True,
    },
}
REPORT_PATH = PROJECT_DIR / "runs" / "clean_retrieval_stage2_report.json"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in report.items() if k != "runs"}, indent=2))
print("Saved:", REPORT_PATH)

## Interactive structured retrieval

At this stage the caller supplies structured constraints explicitly. In Stage 3,
the already validated model planner will produce this same JSON contract.

In [ ]:
my_request = {
    "query": "an elegant woody fragrance for autumn evenings, but no oud",
    "top_k": 10,
    "filters": {"season": "autumn", "time": "night"},
    "wanted_terms": ["woody"],
    "exclude_terms": ["oud"],
}
my_results = post_json("/search", my_request)
for index, item in enumerate(my_results["results"], 1):
    print(f"{index:02d}. {item['label']} | score={item['score']:.3f}")
print("Elapsed:", my_results["elapsed_seconds"], "seconds")